# 🧠 MindGuard AI: Core Model Training Pipeline
**Author:** Mohit Parmar

This notebook demonstrates how to fine-tune the multilingual **XLM-RoBERTa** model to classify psychological text into specific mental health emotional states and risk levels. 

**Requirements:**
1. Go to `Runtime` -> `Change runtime type` and select **T4 GPU**.
2. Upload your cleaned `master_training_data.csv` to the main Colab files folder before running.

In [ ]:
# Install the required Hugging Face and ML libraries
!pip install transformers[torch] accelerate datasets scikit-learn pandas

### Step 1: Preprocessing the Dataset
We will load the data, remove any corrupted rows, and convert our text labels (e.g., 'Anxiety') into numerical IDs that the neural network can understand.

In [ ]:
import pandas as pd
import os
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset

print("🚀 Initializing MindGuard Training Pipeline...")

# 1. Setup Paths
data_path = "master_training_data.csv"
artifacts_dir = "./artifacts/xlmr_weights"
os.makedirs(artifacts_dir, exist_ok=True)

# 2. Load and Clean Data
# 1. Drop any rows where the label is just a number (e.g., "0" or "1")
df = df[~df['label'].str.isnumeric()]
# 2. Drop the corrupted 'admi' label
df = df[df['label'] != 'admi']

# 3. Encode Labels
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])
num_labels = len(label_encoder.classes_)

# Save the label mapping for future inference
mapping = dict(zip(label_encoder.transform(label_encoder.classes_), label_encoder.classes_))
print(f"Detected {num_labels} unique emotions: {mapping}")

# 4. Train/Test Split (80% Training, 20% Validation)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

### Step 2: Tokenizing the Text
XLM-RoBERTa does not understand English words. We use its specific tokenizer to convert our sentences into mathematical tokens, padding them so they are all exactly 128 tokens long.

In [ ]:
from transformers import XLMRobertaTokenizer

print("Loading XLM-RoBERTa Tokenizer...")
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

print("Tokenizing the datasets...")
# Apply tokenization
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# Format the data strictly for PyTorch (removing raw text to avoid tensor errors)
tokenized_train = tokenized_train.rename_column("label_encoded", "labels")
tokenized_val = tokenized_val.rename_column("label_encoded", "labels")

tokenized_train = tokenized_train.remove_columns(["text", "label"])
tokenized_val = tokenized_val.remove_columns(["text", "label"])

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

### Step 3: Model Configuration & Training
We load the base model and configure the `Trainer`. Because we are using a cloud GPU, we can utilize `fp16=True` (Mixed Precision) to train twice as fast, and increase our batch size to 32.

In [ ]:
from transformers import XLMRobertaForSequenceClassification, Trainer, TrainingArguments

print("Loading XLM-RoBERTa Neural Network...")
model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=num_labels)

# Configure Training Rules
training_args = TrainingArguments(
    output_dir=artifacts_dir,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32, # Optimized for 16GB VRAM GPUs (like Colab T4)
    num_train_epochs=3,             # 3 epochs is the sweet spot for transfer learning
    weight_decay=0.01,
    save_strategy="epoch",
    fp16=True                       # Enables half-precision for faster GPU training
)

# Initialize the Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

print("🔥 Starting GPU model training!")
trainer.train()

# Save the final trained model and tokenizer
final_model_dir = os.path.join(artifacts_dir, "final_mindguard_model")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"✅ Training complete. Model saved to {final_model_dir}")

### Step 4: Download the Weights
Run this final cell to zip your newly trained model so you can download it to your local machine for inference and UI integration.

In [ ]:
# Compress the artifacts folder for easy downloading
!zip -r mindguard_model.zip ./artifacts/xlmr_weights/final_mindguard_model/

CODE WE NEED TO RUN IN COLAB/KAGGLE B/C DATASET IS IMBALANCED

In [ ]:
# --- MINDGUARD AI: KAGGLE PRODUCTION PIPELINE ---

import pandas as pd
import numpy as np
import os
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

print("🚀 Initializing Kaggle MindGuard Pipeline...")

# 1. Bulletproof Kaggle Paths
# Kaggle dynamically names input folders, so we search for your CSV automatically!
data_path = ""
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith(".csv"):
            data_path = os.path.join(dirname, filename)
            print(f"📁 Found data at: {data_path}")

# Kaggle requires all saved files to go into the /kaggle/working/ directory
artifacts_dir = "/kaggle/working/artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

# 2. Load and Sanitize Data
print("🧹 Cleaning data...")
df = pd.read_csv(data_path, on_bad_lines='skip')
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str)
df['label'] = df['label'].astype(str)

# --- THE SANITIZER ---
df = df[~df['label'].str.isnumeric()]
df = df[df['label'] != 'admi']

# 3. Label Encoding
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])
num_labels = len(label_encoder.classes_)

mapping = dict(zip(label_encoder.transform(label_encoder.classes_), label_encoder.classes_))
print(f"✅ Detected {num_labels} unique emotions.")

# 4. Train/Test Split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# 5. Calculate Penalty Weights (For Imbalanced Data)
print("⚖️ Calculating Class Weights...")
unique_classes = np.unique(train_df['label_encoded'])
weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=train_df['label_encoded'])
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to('cuda' if torch.cuda.is_available() else 'cpu')

# 6. Tokenization
print("🤖 Loading Tokenizer...")
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.rename_column("label_encoded", "labels")
tokenized_val = tokenized_val.rename_column("label_encoded", "labels")
tokenized_train = tokenized_train.remove_columns(["text", "label"])
tokenized_val = tokenized_val.remove_columns(["text", "label"])
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

# 7. Model Loading
print("🧠 Loading XLM-RoBERTa...")
model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=num_labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='macro') 
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1_macro': f1}

training_args = TrainingArguments(
    output_dir=artifacts_dir,
    eval_strategy="epoch",
    learning_rate=3e-5,              
    per_device_train_batch_size=32, 
    num_train_epochs=5,              
    warmup_steps=500,                
    weight_decay=0.01,
    save_strategy="epoch",
    fp16=True,                       
    metric_for_best_model="f1_macro", 
    load_best_model_at_end=True,
    report_to="none" # Prevents Kaggle from trying to log into external tracking websites
)

# Custom Trainer Override for Imbalanced Data
class ImbalancedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 8. Start Training
trainer = ImbalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

print("🔥 Starting OPTIMIZED Kaggle GPU model training! (5 Epochs)...")
trainer.train()

# 9. Save & Export
final_model_dir = os.path.join(artifacts_dir, "final_mindguard_model")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"✅ Training complete. Model saved to {final_model_dir}")

print("📦 Zipping model for download...")
# Zips the file directly into the /kaggle/working/ directory so you can click it!
!zip -r -q /kaggle/working/mindguard_model.zip {final_model_dir}
print("🎉 Done! Look at the 'Output' panel on the right to download your zip file.")